In [98]:
import os
import json
import sqlite3
import pandas as pd
import re
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.style import WD_STYLE_TYPE
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

AGENT_MODEL = "gpt-4o"

print("Imports loaded")
print(f"Model: {AGENT_MODEL}")

Imports loaded
Model: gpt-4o


In [99]:
# ── Load Classified Requirements from Registry ────────────────────────────────
def load_requirements_from_registry(
    db_path      = '../data/requirements_registry.db',
    project_name = None
):
    """
    Load all classified requirements from the SQLite registry.
    Groups them by label for structured SRS generation.
    """
    conn = sqlite3.connect(db_path)

    query = '''
        SELECT final_text, label, confidence, status
        FROM requirements
        WHERE project_name = ?
        AND label NOT IN ("Ambiguous")
    '''
    rows = conn.execute(query, (project_name,)).fetchall()
    conn.close()

    df = pd.DataFrame(rows, columns=['text', 'label', 'confidence', 'status'])

    # Deduplicate by text
    before = len(df)
    df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
    after = len(df)
    if before != after:
        print(f"Removed {before - len(df)} duplicate requirements")

    # Group by label
    grouped = {}
    for label in df['label'].unique():
        grouped[label] = df[df['label'] == label]['text'].tolist()

    print(f"Total requirements loaded: {len(df)}")
    print(f"\nBy category:")
    for label, reqs in grouped.items():
        count = len(reqs)
        print(f"  {label:<25}: {count}")

    return df, grouped

# Load all classified requirements for SRS generation
df, grouped_requirements = load_requirements_from_registry(
    project_name="Demo Project"
)

Total requirements loaded: 9

By category:
  NFR_Security             : 2
  NFR_Usability            : 2
  FR                       : 1
  NFR_Performance          : 1
  NFR_Reliability          : 2
  NFR_Other                : 1


In [100]:
# ── Collect Project Context for SRS Header ────────────────────────────────
def collect_project_context():
    """
    Collect basic project information needed for SRS header sections.
    In production this would come from a UI form.
    For notebook use, we define it as a dict directly.
    """
    context = {
        "project_name"    : "AI Requirements Elicitation System",
        "version"         : "1.0",
        "date"            : datetime.now().strftime("%B %d, %Y"),
        "organization"    : "Faculty of Computer Science",
        "authors"         : ["Alan Wong"],
        "document_status" : "Draft",
        "description"     : (
            "An AI-powered agent that classifies software requirements "
            "as functional or non-functional, elicits clarification for "
            "ambiguous inputs, and generates structured SRS documents."
        ),
        "intended_users"  : [
            "Software developers",
            "Requirements engineers",
            "Project managers",
            "Academic researchers"
        ],
        "scope": (
            "This system assists requirements engineers by automating "
            "the classification of stakeholder inputs into functional "
            "and non-functional requirement categories, reducing manual "
            "effort in the requirements elicitation phase."
        )
    }
    return context

context = collect_project_context()
print("Project context loaded")
print(f" Project : {context['project_name']}")
print(f" Version : {context['version']}")
print(f" Date    : {context['date']}")

Project context loaded
 Project : AI Requirements Elicitation System
 Version : 1.0
 Date    : April 06, 2026


In [106]:
# ── SRS Section Generation Functions ────────────────────────────────
def strip_gpt_section_header(text, section_number):
    """
    Remove the first line of GPT output if it repeats the section heading.
    e.g. removes "1. Introduction" or "2. Overall Description".
    """
    lines = text.strip().split('\n')
    if not lines:
        return text
    
    first_line = lines[0].strip()
    
    # Remove if first line starts with the section number
    # Catches: "1. Introduction", "2. Overall Description", "3. Specific Requirements"
    if re.match(rf'^{re.escape(str(section_number))}[\.\s]', first_line):
        lines = lines[1:]  # drop first line
    
    # Also remove any leading blank lines after stripping
    while lines and not lines[0].strip():
        lines = lines.pop(0)
    
    return '\n'.join(lines).strip()

def generate_introduction_section(context, grouped):
    """Generate Section 1 — Introduction using GPT"""

    total_reqs = sum(len(v) for v in grouped.values())
    fr_count   = len(grouped.get('FR', []))
    nfr_count  = total_reqs - fr_count

    prompt = f"""You are a software requirements engineer writing a formal SRS document.

Write Section 1 (Introduction) of an IEEE 830 Software Requirements Specification.

Project details:
- Name        : {context['project_name']}
- Description : {context['description']}
- Scope       : {context['scope']}
- Intended users: {', '.join(context['intended_users'])}
- Total requirements: {total_reqs} ({fr_count} functional, {nfr_count} non-functional)

Write these subsections in clear professional technical writing:
1.1 Purpose
1.2 Scope
1.3 Definitions and Abbreviations
1.4 Intended Audience
1.5 Document Overview

IMPORTANT: Do NOT include "1. Introduction" as a heading — start directly
with the introductory paragraph followed by the subsections.

Use formal IEEE 830 language. Be concise but complete.
Do NOT use markdown formatting — write in plain paragraphs."""

    response = client.chat.completions.create(
        model      = AGENT_MODEL,
        messages   = [
            {"role": "system", "content": "You are a technical writer producing IEEE 830 SRS documents."},
            {"role": "user",   "content": prompt}
        ],
        max_tokens  = 1000,
        temperature = 0.2
    )
    return response.choices[0].message.content.strip()


def generate_overall_description(context):
    """Generate Section 2 — Overall Description"""

    prompt = f"""Write Section 2 (Overall Description) of an IEEE 830 SRS document.

Project: {context['project_name']}
Description: {context['description']}

Include these subsections:
2.1 Product Perspective
2.2 Product Functions (high-level summary only)
2.3 User Characteristics
2.4 Constraints
2.5 Assumptions and Dependencies

IMPORTANT: Do NOT include "2. Overall Description" as a heading — start directly
with the introductory paragraph followed by the subsections.

Use formal IEEE 830 language. Plain paragraphs only — no markdown."""

    response = client.chat.completions.create(
        model      = AGENT_MODEL,
        messages   = [
            {"role": "system", "content": "You are a technical writer producing IEEE 830 SRS documents."},
            {"role": "user",   "content": prompt}
        ],
        max_tokens  = 800,
        temperature = 0.2
    )
    return response.choices[0].message.content.strip()


def generate_functional_requirements(fr_list):
    """Generate Section 3.1 — Functional Requirements with IDs"""

    if not fr_list:
        return "No functional requirements have been classified."

    reqs_text = '\n'.join([f"FR-{str(i+1).zfill(3)}: {req}" for i, req in enumerate(fr_list)])

    prompt = f"""You are writing Section 3.1 (Functional Requirements) of an IEEE 830 SRS.

You have EXACTLY {len(fr_list)} functional requirements to document.
Do NOT add, split, merge, or invent any requirements.
Document ONLY the {len(fr_list)} requirements listed below — no more, no less.
    
Given these classified functional requirements:
{reqs_text}

For each requirement:
1. Assign a unique ID: FR-001, FR-002, etc.
2. Write a clear Description
3. Assign Priority: High / Medium / Low
4. Write one Acceptance Criterion (how to verify it)

Format each requirement EXACTLY like this:
FR-001
Description: [requirement text]
Priority: High
Acceptance Criterion: [how to test/verify this requirement]

No markdown. No bullet points. Separate each requirement with a blank line."""

    response = client.chat.completions.create(
        model      = AGENT_MODEL,
        messages   = [
            {"role": "system", "content": (
                "You are a requirements engineer writing formal SRS documentation."
                "Document ONLY the requirements provided. Never add new ones."
                )
            },
            {"role": "user",   "content": prompt}
        ],
        max_tokens  = 2000,
        temperature = 0
    )
    return response.choices[0].message.content.strip()


def generate_nfr_section(grouped):
    """Generate Section 3.2 — Non-Functional Requirements grouped by category"""

    nfr_categories = {
        k: v for k, v in grouped.items()
        if k.startswith('NFR_') and v
    }

    if not nfr_categories:
        return "No non-functional requirements have been classified."

    # Category display names
    category_names = {
        'NFR_Performance'     : '3.2.1 Performance Requirements',
        'NFR_Security'        : '3.2.2 Security Requirements',
        'NFR_Usability'       : '3.2.3 Usability Requirements',
        'NFR_Reliability'     : '3.2.4 Reliability Requirements',
        'NFR_Maintainability' : '3.2.5 Maintainability Requirements',
        'NFR_Scalability'     : '3.2.6 Scalability Requirements',
        'NFR_Portability'     : '3.2.7 Portability Requirements',
        'NFR_Operational'     : '3.2.8 Operational Requirements',
        'NFR_Legal'           : '3.2.9 Legal and Compliance Requirements',
        'NFR_LookAndFeel'     : '3.2.10 Look and Feel Requirements',
        'NFR_Other'           : '3.2.11 Other Non-Functional Requirements',
    }

    all_nfr_text = ""

    for category, reqs in nfr_categories.items():
        if not reqs:
            continue

        section_name = category_names.get(category, category)
        prefix       = category
        reqs_text = '\n'.join([f"{prefix}-{str(i+1).zfill(3)}: {req}" for i, req in enumerate(reqs)])

        prompt = f"""Write the {section_name} subsection of an IEEE 830 SRS.

You have EXACTLY {len(reqs)} requirements to document.
Do NOT add, split, merge, or invent any requirements.
Document ONLY these {len(reqs)} requirements — no more, no less.

Requirements to document:
{reqs_text}

For each requirement assign:
- ID: {prefix}-001, {prefix}-002, etc.
- Description
- Measurable target (infer reasonable values where missing)
- Verification method

Format:
{prefix}-001
Description: [text]
Target: [measurable criterion]
Verification: [how to verify]

CRITICAL FORMATTING RULES:
1. The ID line ({prefix}-001) must be on its own line with NO bullet, NO dash, NO dot before it
2. Do NOT put a bullet point or dash before any ID line
3. Separate each requirement with exactly one blank line
4. No markdown, no asterisks, no special characters before IDs

Do NOT include the section heading "{section_name}" in your response."""

        response = client.chat.completions.create(
            model      = AGENT_MODEL,
            messages   = [
                {"role": "system", "content": (
                    "You are a requirements engineer writing formal SRS documentation."
                    "Document ONLY the requirements provided. Never add new ones."
                    )
                },
                {"role": "user",   "content": prompt}
            ],
            max_tokens  = 1000,
            temperature = 0
        )

        all_nfr_text += f"\n{section_name}\n\n"
        all_nfr_text += response.choices[0].message.content.strip()
        all_nfr_text += "\n\n"

    return all_nfr_text

print("Section generator functions defined")

Section generator functions defined


In [107]:
# ── Deduplicate Registry Entries ────────────────────────────────
def deduplicate_registry(db_path='../data/requirements_registry.db'):
    conn = sqlite3.connect(db_path)
    
    # Check current state
    total = conn.execute('SELECT COUNT(*) FROM requirements').fetchone()[0]
    print(f"Before dedup: {total} rows")
    
    # Keep only the most recent entry per unique text + project combination
    conn.execute('''
        DELETE FROM requirements
        WHERE id NOT IN (
            SELECT MAX(id)
            FROM requirements
            GROUP BY final_text, project_name
        )
    ''')
    conn.commit()
    
    remaining = conn.execute('SELECT COUNT(*) FROM requirements').fetchone()[0]
    print(f"After dedup : {remaining} rows")
    print(f"Removed     : {total - remaining} duplicates")
    conn.close()

deduplicate_registry()

Before dedup: 9 rows
After dedup : 9 rows
Removed     : 0 duplicates


In [108]:
# ── Generate SRS Document Sections ────────────────────────────────
print("Generating SRS document sections...\n")

print("[1/4] Generating Introduction...")
section1 = generate_introduction_section(context, grouped_requirements)
section1 = strip_gpt_section_header(section1, 1)
print("Done")

print("[2/4] Generating Overall Description...")
section2 = generate_overall_description(context)
section2 = strip_gpt_section_header(section2, 2)
print("Done")

print("[3/4] Generating Functional Requirements...")
fr_list  = grouped_requirements.get('FR', [])
section3_fr = generate_functional_requirements(fr_list)
print(f"Done ({len(fr_list)} FRs)")

print("[4/4] Generating Non-Functional Requirements...")
section3_nfr = generate_nfr_section(grouped_requirements)
print("Done")

print("\nAll sections generated")

# Preview first 500 chars of each section
print("\n=== SECTION PREVIEWS ===")
for name, content in [
    ("Section 1.0", section1),
    ("Section 2.0", section2),
    ("Section 3.1 FR", section3_fr),
    ("Section 3.2 NFR", section3_nfr)
]:
    print(f"\n--- {name} ---")
    print(content[:300] + "...")

Generating SRS document sections...

[1/4] Generating Introduction...
Done
[2/4] Generating Overall Description...
Done
[3/4] Generating Functional Requirements...
Done (1 FRs)
[4/4] Generating Non-Functional Requirements...
Done

All sections generated

=== SECTION PREVIEWS ===

--- Section 1.0 ---
This Software Requirements Specification (SRS) document outlines the requirements for the AI Requirements Elicitation System. This system is designed to enhance the efficiency and accuracy of the requirements elicitation process by leveraging artificial intelligence to classify software requirements...

--- Section 2.0 ---
The AI Requirements Elicitation System is designed to streamline the process of software requirements specification by leveraging artificial intelligence to classify, clarify, and document requirements. This system aims to enhance the efficiency and accuracy of requirements engineering by automating...

--- Section 3.1 FR ---
FR-001  
Description: The system shall allow u

In [111]:
# ── Build SRS .docx Document ────────────────────────────────
def is_requirement_id(line):
    """
    Detect requirement ID lines like:
    FR-001, NFR_Security-001, NFRSecurity-001, NFR_Other-001
    Strips leading bullets or special chars before checking.
    """
    cleaned = line.strip().lstrip('·•-– \t')
    return bool(re.match(
        r'^(FR|NFR[_]?[A-Za-z]*)-\d{3}$',
        cleaned.strip()
    ))

def clean_id_line(line):
    """Strip any leading bullet characters from ID lines"""
    return line.strip().lstrip('·•-– \t').strip()

def build_srs_docx(context, section1, section2, section3_fr, section3_nfr):
    """
    Build a professionally formatted IEEE 830 SRS .docx file.
    """
    doc = Document()

    # ── Page margins ──────────────────────────────────────────────────────
    for section in doc.sections:
        section.top_margin    = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin   = Inches(1.25)
        section.right_margin  = Inches(1.25)

    # ── Helper functions ──────────────────────────────────────────────────
    def add_heading(text, level=1):
        heading = doc.add_heading(text, level=level)
        heading.alignment = WD_ALIGN_PARAGRAPH.LEFT
        run = heading.runs[0]
        run.font.color.rgb = RGBColor(0x1F, 0x49, 0x7D)
        return heading

    def add_paragraph(text, bold=False, size=11):
        para = doc.add_paragraph()
        run  = para.add_run(text)
        run.font.size = Pt(size)
        run.font.bold = bold
        return para

    def add_requirement_block(text):
        """Add a formatted requirement block with gray background"""
        lines = text.strip().split('\n')
        for line in lines:
            if not line.strip():
                doc.add_paragraph()
                continue
            # Requirement ID line (e.g. FR-001)
            if line.strip() and ':' not in line and len(line.strip()) < 15:
                p    = doc.add_paragraph()
                run  = p.add_run(line.strip())
                run.font.bold      = True
                run.font.size      = Pt(11)
                run.font.color.rgb = RGBColor(0x1F, 0x49, 0x7D)
            else:
                para = doc.add_paragraph(style='List Bullet')
                para.paragraph_format.left_indent = Inches(0.25)
                run  = para.add_run(line.strip())
                run.font.size = Pt(10.5)

    # ── Title Page ────────────────────────────────────────────────────────
    title = doc.add_heading('', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = title.add_run('Software Requirements Specification')
    run.font.size      = Pt(24)
    run.font.bold      = True
    run.font.color.rgb = RGBColor(0x1F, 0x49, 0x7D)

    doc.add_paragraph()

    subtitle = doc.add_paragraph()
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = subtitle.add_run(context['project_name'])
    run.font.size = Pt(16)
    run.font.bold = True

    doc.add_paragraph()

    # Metadata table
    meta_table = doc.add_table(rows=4, cols=2)
    meta_table.style = 'Table Grid'
    meta_data = [
        ('Version',         context['version']),
        ('Date',            context['date']),
        ('Status',          context['document_status']),
        ('Organization',    context['organization']),
    ]
    for i, (key, value) in enumerate(meta_data):
        meta_table.rows[i].cells[0].text = key
        meta_table.rows[i].cells[1].text = value
        meta_table.rows[i].cells[0].paragraphs[0].runs[0].font.bold = True

    doc.add_page_break()

    # ── Table of Contents placeholder ─────────────────────────────────────
    add_heading('Table of Contents', level=1)
    toc_items = [
        '1.0 Introduction',
        '    1.1 Purpose',
        '    1.2 Scope',
        '    1.3 Definitions and Abbreviations',
        '    1.4 Intended Audience',
        '    1.5 Document Overview',
        '2.0 Overall Description',
        '    2.1 Product Perspective',
        '    2.2 Product Functions',
        '    2.3 User Characteristics',
        '    2.4 Constraints',
        '    2.5 Assumptions and Dependencies',
        '3.0 Specific Requirements',
        '    3.1 Functional Requirements',
        '    3.2 Non-Functional Requirements',
    ]
    for item in toc_items:
        add_paragraph(item)
    doc.add_page_break()

    # ── Section 1: Introduction ───────────────────────────────────────────
    add_heading('1.0 Introduction', level=1)
    for line in section1.split('\n'):
        if not line.strip():
            continue
        if line.strip().startswith('1.'):
            add_heading(line.strip(), level=2)
        else:
            add_paragraph(line.strip())
    doc.add_page_break()

    # ── Section 2: Overall Description ───────────────────────────────────
    add_heading('2.0 Overall Description', level=1)
    for line in section2.split('\n'):
        if not line.strip():
            continue
        if line.strip().startswith('2.'):
            add_heading(line.strip(), level=2)
        else:
            add_paragraph(line.strip())
    doc.add_page_break()

    # ── Section 3: Specific Requirements ─────────────────────────────────
    add_heading('3.0 Specific Requirements', level=1)

    add_heading('3.1 Functional Requirements', level=2)
    add_paragraph(
        f"This section defines {len(fr_list)} functional requirements "
        f"elicited and classified by the AI Requirements Elicitation Agent.",
        bold=False
    )
    doc.add_paragraph()
    
    for line in section3_fr.split('\n'):
        stripped = line.strip()

        if not stripped:
            doc.add_paragraph()
            continue

        # FR ID line
        elif is_requirement_id(stripped):
            clean_id = clean_id_line(stripped)
            p   = doc.add_paragraph()
            run = p.add_run(clean_id)
            run.font.bold      = True
            run.font.size      = Pt(11)
            run.font.color.rgb = RGBColor(0x1F, 0x49, 0x7D)

        # Detail lines
        else:
            para = doc.add_paragraph(style='List Bullet')
            para.paragraph_format.left_indent = Inches(0.25)
            run  = para.add_run(stripped.lstrip('·•-- \t'))
            run.font.size = Pt(10.5)

    doc.add_paragraph()
    
    add_heading('3.2 Non-Functional Requirements', level=2)
    for line in section3_nfr.split('\n'):
        stripped = line.strip()

        if not stripped:
            doc.add_paragraph()
            continue

        # Section heading e.g. "3.2.1 Performance Requirements"
        if re.match(r'^3\.2\.\d+', stripped):
            add_heading(stripped, level=3)

        # Requirement ID line e.g. "NFR_Security-001" or "· NFR_Security-001"
        elif is_requirement_id(stripped):
            clean_id = clean_id_line(stripped)
            p   = doc.add_paragraph()
            run = p.add_run(clean_id)
            run.font.bold      = True
            run.font.size      = Pt(11)
            run.font.color.rgb = RGBColor(0x1F, 0x49, 0x7D)

        # Detail lines e.g. "Description: ..." "Target: ..." "Verification: ..."
        else:
            para = doc.add_paragraph(style='List Bullet')
            para.paragraph_format.left_indent = Inches(0.25)
            run  = para.add_run(stripped.lstrip('·•-- \t'))
            run.font.size = Pt(10.5)

    # ── Footer ────────────────────────────────────────────────────────────
    doc.add_page_break()
    footer_para = doc.add_paragraph()
    footer_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = footer_para.add_run(
        f"Generated by AI Requirements Elicitation Agent — {context['date']}"
    )
    run.font.size   = Pt(9)
    run.font.italic = True
    run.font.color.rgb = RGBColor(0x88, 0x88, 0x88)

    return doc

# Build document
doc = build_srs_docx(context, section1, section2, section3_fr, section3_nfr)

# Save
output_path = f"../outputs/SRS_{context['project_name'].replace(' ', '_')}_v{context['version']}.docx"
os.makedirs('../outputs', exist_ok=True)
doc.save(output_path)

print(f"SRS document saved to:")
print(f"{output_path}")

SRS document saved to:
../outputs/SRS_AI_Requirements_Elicitation_System_v1.0.docx


In [112]:
# ── Full Pipeline Function ────────────────────────────────
def generate_srs_from_registry(
    project_name,
    db_path      = '../data/requirements_registry.db',
    context      = None
):
    """
    Full pipeline — loads requirements from registry
    and generates a complete SRS .docx file.
    Call this from the agent or Streamlit UI.
    """
    print(f"Generating SRS for: {project_name}\n")

    # Load requirements
    df, grouped = load_requirements_from_registry(db_path, project_name)

    if df.empty:
        print("No classified requirements found in registry.")
        return None

    if context is None:
        context = collect_project_context()

    # Generate sections
    print("Generating sections...")
    s1      = generate_introduction_section(context, grouped)
    s2      = generate_overall_description(context)
    s3_fr   = generate_functional_requirements(grouped.get('FR', []))
    s3_nfr  = generate_nfr_section(grouped)

    # Build and save document
    fr_list     = grouped.get('FR', [])
    doc         = build_srs_docx(context, s1, s2, s3_fr, s3_nfr)
    output_path = (
        f"../outputs/SRS_{project_name.replace(' ', '_')}.docx"
    )
    os.makedirs('../outputs', exist_ok=True)
    doc.save(output_path)

    print(f"\nSRS generated successfully")
    print(f"File    : {output_path}")
    print(f"FRs     : {len(fr_list)}")
    print(f"NFR cats: {len([k for k in grouped if k.startswith('NFR_')])}")

    return output_path

# Test the full pipeline
output = generate_srs_from_registry("Demo Project")

Generating SRS for: Demo Project

Total requirements loaded: 9

By category:
  NFR_Security             : 2
  NFR_Usability            : 2
  FR                       : 1
  NFR_Performance          : 1
  NFR_Reliability          : 2
  NFR_Other                : 1
Generating sections...

SRS generated successfully
File    : ../outputs/SRS_Demo_Project.docx
FRs     : 1
NFR cats: 5
